
# Exploração dos dados

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
df = spark.sql(
    '''
    select * from workspace.silver_layer.online_retail_transactions
    '''
)

df.display()

In [0]:
df.filter(col('stock_code') == '84519B').display()

In [0]:
df.filter(col('description') == 'Discount').display()

In [0]:
df.filter(col('quantity') <= 0).display()


## Análises

1. Quantidade de clientes únicos

2. Período disponível
   - menor invoice_date
   - maior invoice_date

3. Quantidade de compras por cliente

4. Distribuição da recência
   - dias desde a última compra

5. Intervalo entre compras

6. Distribuição do valor comprado
   - total_amount
   - ticket por invoice

7. Cancelamentos e devoluções
   - volume
   - percentual
   - impacto financeiro

8. Quantidade de clientes por país

9. Evolução mensal
   - clientes ativos
   - pedidos
   - receita

10. Quantidade de clientes que voltam após:
   - 30 dias
   - 60 dias
   - 90 dias


### Quantidade de clientes únicos



In [0]:
df.agg(countDistinct('customer_id')).display()


### Período disponível

In [0]:
df.agg(min('invoice_date'), max('invoice_date')).display()


### Quantidade de compras por cliente

In [0]:
(
    df
    .groupBy('customer_id')
    .agg(
        count('*').alias('num_transactions'),
        avg('quantity').alias('avg_quantity'),
        (sum('quantity') * sum('price')).alias('total_amount')
    )
    .orderBy(col('avg_quantity').desc())
    .display()
)


### Intervalo entre compras

In [0]:
window_spec = Window.partitionBy('customer_id').orderBy('invoice_date')

df_with_lag = df.withColumn('prev_invoice_date', lag('invoice_date').over(window_spec))

display(
    df_with_lag
    .withColumn(
        'days_between_invoices',
        datediff(col('invoice_date'), col('prev_invoice_date'))
        )
    .withColumn(
        'date_invoice_date', 
        date_format(col('invoice_date'), 'yyyy-MM-dd')
        )
    .withColumn(
        'date_prev_invoice_date', 
        date_format(col('prev_invoice_date'), 'yyyy-MM-dd')
        )
    .filter(col('date_invoice_date') != col('date_prev_invoice_date'))
    .select(
        'customer_id', 'days_between_invoices' 
        )
)

Databricks visualization. Run in Databricks to view.


### Cancelamentos e devoluções

In [0]:
(
    df
    .withColumn(
        "is_cancelled_v2",
        col("invoice").startswith("C")
    )
    .groupby('is_cancelled_v2')
    .agg(
        count('*').alias('num_transactions'), 
        sum('quantity').alias('total_quantity'), 
        (sum('quantity') * sum('price')).alias('total_price')
    )
    .display()
)


### Variável resposta: target_reactivated_30d

- Já fez alguma compra válida no passado (sem devolução)
- Está há pelo menos 60 dias sem comprar
- Comprou 30 dias depois

In [0]:
df_with_filters = (
    df_with_lag
    .withColumn(
        "is_cancelled_v2",
        col("invoice").startswith("C")
    )
    .withColumn(
        "is_test",
        col("stock_code").startswith("TEST")
    )
     .withColumn(
        "is_discount",
        col("description") == lit('Discount')
    )
    .filter(col('is_cancelled_v2') == False)
    .filter(col('is_test') == False)
    .filter(col('price') > 0)
    .filter(col('quantity') > 0)
    .filter(col('is_discount') == False)
    # .display()
)

In [0]:
reference_date = "2011-11-08"

df_last_purchase = (
    df_with_filters
    .withColumn(
        "purchase_date",
        to_date("invoice_date")
    )
    .filter(
        col("purchase_date") <= lit(reference_date)
    )
    .groupBy(
        "customer_id"
    )
    .agg(
        max("purchase_date").alias(
            "last_purchase_date"
        )
    )
    .withColumn(
        "reference_date",
        lit(reference_date).cast("date")
    )
    .withColumn(
        "inactive_days",
        datediff(
            col("reference_date"),
            col("last_purchase_date")
        )
    )
    .filter(
        col("inactive_days") >= 60
    )
)


# Clientes que compraram nos próximos 30 dias
df_reactivated = (
    df_with_filters
    .withColumn(
        "purchase_date",
        to_date("invoice_date")
    )
    .filter(
        (col("purchase_date") > lit(reference_date))
        &
        (
            col("purchase_date")
            <= date_add(
                lit(reference_date),
                30
            )
        )
    )
    .select(
        "customer_id"
    )
    .distinct()
    .withColumn(
        "target_reactivated_30d",
        lit(1)
    )
)


# Criação da variável resposta
df_target = (
    df_last_purchase
    .join(
        df_reactivated,
        on="customer_id",
        how="left"
    )
    .fillna(
        {
            "target_reactivated_30d": 0
        }
    )
)

df_target.display()


## Novas variáveis 

- Excluir stock_code = TEST001
- total_amount (ver alguns casos nulos)
- is_zero_price
- is_discount
- is_cancellation (start with C)
- is_return -> Variável resposta
- Preço médio (total_amount) por país 
- Preço médio (total_amount) por stock_code
